In [1]:
import json
import numpy as np
import pandas as pd

from pathlib import Path

RAW_DIR = Path("..") / "data" / "raw"

FILES = {
    "papers": RAW_DIR / "starrydata_papers.csv",
    "samples": RAW_DIR / "starrydata_samples.csv",
    "curves": RAW_DIR / "starrydata_curves.csv",
}

print("Working directory:", Path.cwd())

for name, path in FILES.items():
    print(f"{name:8} exists={path.exists()} -> {path.resolve()}")

Working directory: c:\Users\adity\OneDrive\Desktop\ADSA\Data_Managment\Group Project\DM_Project_Group_Data-Managers\notebooks
papers   exists=True -> C:\Users\adity\OneDrive\Desktop\ADSA\Data_Managment\Group Project\DM_Project_Group_Data-Managers\data\raw\starrydata_papers.csv
samples  exists=True -> C:\Users\adity\OneDrive\Desktop\ADSA\Data_Managment\Group Project\DM_Project_Group_Data-Managers\data\raw\starrydata_samples.csv
curves   exists=True -> C:\Users\adity\OneDrive\Desktop\ADSA\Data_Managment\Group Project\DM_Project_Group_Data-Managers\data\raw\starrydata_curves.csv


Load the complete datasets

In [2]:
papers = pd.read_csv(FILES["papers"], low_memory=False)
samples = pd.read_csv(FILES["samples"], low_memory=False)
curves = pd.read_csv(FILES["curves"], low_memory=False)

datasets = {
    "papers": papers,
    "samples": samples,
    "curves": curves
}

print("All datasets loaded successfully!")

All datasets loaded successfully!


Calculate the dataset dimensions

In [3]:
dataset_summary = []

for name, df in datasets.items():
    dataset_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Total Cells": df.size
    })

summary_df = pd.DataFrame(dataset_summary)

display(summary_df)

,Dataset,Rows,Columns,Total Cells
0,papers,56495,15,847425
1,samples,106331,9,956979
2,curves,237313,16,3797008


### 4. Missing Values Analysis

In [4]:
missing_results = []

for name, df in datasets.items():

    for column in df.columns:

        null_count = df[column].isna().sum()

        missing_percentage = (
            null_count / len(df) * 100
            if len(df) > 0 else 0
        )

        missing_results.append({
            "Dataset": name,
            "Column": column,
            "Missing_Count": null_count,
            "Missing_Percentage": round(missing_percentage, 2)
        })

missing_df = pd.DataFrame(missing_results)

display(
    missing_df.sort_values(
        by="Missing_Percentage",
        ascending=False
    ).reset_index(drop=True)
)

,Dataset,Column,Missing_Count,Missing_Percentage
0,samples,composition_details,87433,82.23
1,curves,comments,103868,43.77
2,papers,issue,3873,6.86
3,papers,page,2266,4.01
4,papers,ISSN,1901,3.36
5,papers,container_title_short,1575,2.79
6,papers,volume,1451,2.57
7,samples,sample_info,2314,2.18
8,samples,composition,191,0.18
9,curves,composition,362,0.15


Initial Missing Value Observations


The initial data profiling revealed that missing values are unevenly distributed across the three datasets. The Samples dataset has a particularly high percentage of missing values in composition_details (82.23%), while the Curves dataset contains 43.77% missing values in comments. The Papers dataset has relatively low missing percentages in its metadata fields. Important identifier fields such as SID and sample_id, along with the numerical curve coordinates x and y, contain no null values in the initial profiling. These findings will guide the data cleaning and exclusion strategy in Phase 3.

Primary key uniqueness

Now we'll test whether the identifiers are actually unique.

In [5]:
def profile_primary_key(df, key, dataset_name):

    if key not in df.columns:
        print(f"{key} not found in {dataset_name}")
        return None

    total_rows = len(df)
    missing_keys = df[key].isna().sum()

    # Count duplicate non-null identifiers
    duplicate_rows = df[key].dropna().duplicated().sum()

    unique_keys = df[key].nunique(dropna=True)

    return {
        "Dataset": dataset_name,
        "Primary Key": key,
        "Total Rows": total_rows,
        "Unique Values": unique_keys,
        "Missing Keys": missing_keys,
        "Duplicate Rows": duplicate_rows,
        "Is Unique": (
            duplicate_rows == 0 and missing_keys == 0
        )
    }


key_results = [
    profile_primary_key(papers, "SID", "Papers"),
    profile_primary_key(samples, "sample_id", "Samples")
]

key_results_df = pd.DataFrame(
    [r for r in key_results if r is not None]
)

display(key_results_df)

,Dataset,Primary Key,Total Rows,Unique Values,Missing Keys,Duplicate Rows,Is Unique
0,Papers,SID,56495,56495,0,0,True
1,Samples,sample_id,106331,106331,0,0,True


### Referential Integrity – Orphan Record Detection

In [6]:
def normalize_ids(series):
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


def check_referential_integrity(
    child_df,
    parent_df,
    child_name
):

    parent_ids = set(
        normalize_ids(parent_df["SID"]).dropna()
    )

    child_ids = normalize_ids(child_df["SID"])

    valid_ids = child_ids.notna()

    orphan_mask = (
        valid_ids & ~child_ids.isin(parent_ids)
    )

    orphan_count = int(orphan_mask.sum())

    return {
        "Dataset": child_name,
        "Total Records": len(child_df),
        "Missing SID": int(child_ids.isna().sum()),
        "Orphan Records": orphan_count,
        "Unique Orphan SIDs": int(
            child_ids[orphan_mask].nunique()
        ),
        "Orphan Percentage": round(
            orphan_count / len(child_df) * 100, 2
        )
    }


referential_results = []

for name, df in [
    ("Samples", samples),
    ("Curves", curves)
]:

    result = check_referential_integrity(
        child_df=df,
        parent_df=papers,
        child_name=name
    )

    referential_results.append(result)


referential_df = pd.DataFrame(referential_results)

display(referential_df)

,Dataset,Total Records,Missing SID,Orphan Records,Unique Orphan SIDs,Orphan Percentage
0,Samples,106331,0,9,3,0.01
1,Curves,237313,0,31,3,0.01


FIND THE ACTUAL ORPHANS SIDs


In [7]:
paper_ids = set(
    normalize_ids(papers["SID"]).dropna()
)

sample_ids = normalize_ids(samples["SID"])
curve_ids = normalize_ids(curves["SID"])

sample_orphan_sids = set(
    sample_ids[
        sample_ids.notna() &
        ~sample_ids.isin(paper_ids)
    ]
)

curve_orphan_sids = set(
    curve_ids[
        curve_ids.notna() &
        ~curve_ids.isin(paper_ids)
    ]
)

print("Sample orphan SIDs:", sample_orphan_sids)
print("Curve orphan SIDs:", curve_orphan_sids)

print(
    "Common orphan SIDs:",
    sample_orphan_sids.intersection(curve_orphan_sids)
)

Sample orphan SIDs: {'8534', '8535', '9026'}
Curve orphan SIDs: {'8535', '8534', '9026'}
Common orphan SIDs: {'8534', '8535', '9026'}


Referential Integrity – Findings


Referential integrity analysis identified 9 orphan records in the Samples dataset, representing 3 unique SIDs, and 31 orphan records in the Curves dataset, also representing 3 unique SIDs. Both datasets contain zero missing SID values. These orphan records reference identifiers that are not present in the Papers dataset. The issue will be investigated before integration, and unresolved orphan records will be excluded from the corresponding integrated output through appropriate join filtering.

JSON parseability analysis

In [8]:
def validate_json(value):

    if pd.isna(value) or not str(value).strip():
        return "Missing"

    try:
        parsed = json.loads(value)

        if isinstance(parsed, dict) and not parsed:
            return "Empty JSON"

        if isinstance(parsed, dict):
            return "Valid JSON"

        return "Valid JSON - Non-dictionary"

    except (json.JSONDecodeError, TypeError):
        return "Invalid JSON"


samples["json_status"] = samples["sample_info"].apply(
    validate_json
)

json_summary = (
    samples["json_status"]
    .value_counts()
    .reset_index()
)

json_summary.columns = ["JSON Status", "Record Count"]

json_summary["Percentage"] = (
    json_summary["Record Count"] / len(samples) * 100
).round(2)

display(json_summary)

,JSON Status,Record Count,Percentage
0,Valid JSON,72456,68.14
1,Empty JSON,31561,29.68
2,Missing,2314,2.18


### JSON Parseability – Findings
The JSON profiling of the sample_info column identified 72,456 valid JSON records (68.14%), 31,561 empty JSON objects (29.68%) and 2,314 missing values (2.18%). No malformed JSON records were detected. Therefore, Python's json.loads() is suitable for extracting nested metadata. Empty and missing JSON values should be handled safely by returning null values for unavailable properties rather than removing the corresponding sample records.

### Data Quality Issue Register

In [9]:

# --------------------------------------------
# Data Quality Issue Register
# --------------------------------------------

issues = []

# 1. Missing values in dataset columns

for _, row in missing_df.iterrows():

    if row["Missing_Count"] == 0:
        continue

    column = row["Column"]
    dataset = row["Dataset"]
    count = int(row["Missing_Count"])
    percentage = row["Missing_Percentage"]

    if percentage >= 50:
        severity = "High"
    elif percentage >= 10:
        severity = "Medium"
    else:
        severity = "Low"

    if column == "composition_details":
        remediation = (
            "Retain available descriptions; preserve missing "
            "values as null. Exclude this field from models "
            "that do not require descriptive text."
        )

    elif column == "comments":
        remediation = (
            "Retain available comments and represent missing "
            "comments as null. Do not discard curve measurements."
        )

    elif column == "sample_info":
        remediation = (
            "Handle missing JSON safely in Python and retain "
            "the original sample record."
        )

    else:
        remediation = (
            "Preserve missing values as null. Evaluate whether "
            "the field is essential before filtering records."
        )

    issues.append({
        "Dataset": dataset,
        "Issue Category": "Missing Values",
        "Identified Issue": f"Missing values in {column}",
        "Affected Records": count,
        "Percentage": percentage,
        "Severity": severity,
        "Proposed Remediation": remediation
    })


# 2. Primary key issues

for _, row in key_results_df.iterrows():

    if row["Missing Keys"] > 0 or row["Duplicate Rows"] > 0:

        issues.append({
            "Dataset": row["Dataset"],
            "Issue Category": "Primary Key",
            "Identified Issue": f"Invalid {row['Primary Key']} values",
            "Affected Records": int(
                row["Missing Keys"] + row["Duplicate Rows"]
            ),
            "Percentage": None,
            "Severity": "High",
            "Proposed Remediation": (
                "Investigate missing or duplicate identifiers "
                "before performing joins."
            )
        })


# 3. Referential integrity issues

for _, row in referential_df.iterrows():

    if row["Orphan Records"] > 0:

        issues.append({
            "Dataset": row["Dataset"],
            "Issue Category": "Referential Integrity",
            "Identified Issue": (
                "Records reference SIDs absent from Papers"
            ),
            "Affected Records": int(row["Orphan Records"]),
            "Percentage": row["Orphan Percentage"],
            "Severity": "High",
            "Proposed Remediation": (
                "Investigate orphan SIDs. Exclude unresolved "
                "records from the integrated output using "
                "appropriate join filtering."
            )
        })


# 4. Empty JSON objects

empty_count = int(
    (samples["json_status"] == "Empty JSON").sum()
)

if empty_count > 0:

    issues.append({
        "Dataset": "Samples",
        "Issue Category": "JSON",
        "Identified Issue": "Empty sample_info JSON objects",
        "Affected Records": empty_count,
        "Percentage": round(empty_count / len(samples) * 100, 2),
        "Severity": "Medium",
        "Proposed Remediation": (
            "Keep sample records and represent unavailable "
            "extracted properties as null."
        )
    })


# 5. Missing JSON

missing_json_count = int(
    (samples["json_status"] == "Missing").sum()
)

if missing_json_count > 0:

    issues.append({
        "Dataset": "Samples",
        "Issue Category": "JSON",
        "Identified Issue": "Missing sample_info JSON",
        "Affected Records": missing_json_count,
        "Percentage": round(
            missing_json_count / len(samples) * 100, 2
        ),
        "Severity": "Low",
        "Proposed Remediation": (
            "Handle missing JSON values safely using "
            "json.loads with missing-value checks."
        )
    })


# 6. Invalid JSON

invalid_json_count = int(
    (samples["json_status"] == "Invalid JSON").sum()
)

if invalid_json_count > 0:

    issues.append({
        "Dataset": "Samples",
        "Issue Category": "JSON",
        "Identified Issue": "Malformed sample_info JSON",
        "Affected Records": invalid_json_count,
        "Percentage": round(
            invalid_json_count / len(samples) * 100, 2
        ),
        "Severity": "Medium",
        "Proposed Remediation": (
            "Investigate and repair malformed JSON before "
            "flattening."
        )
    })


# Create final register

issue_register = pd.DataFrame(issues)

issue_register = issue_register.sort_values(
    by="Affected Records",
    ascending=False
).reset_index(drop=True)

display(issue_register)


,Dataset,Issue Category,Identified Issue,Affected Records,Percentage,Severity,Proposed Remediation
0,curves,Missing Values,Missing values in comments,103868,43.77,Medium,Retain available comments and represent missin...
1,samples,Missing Values,Missing values in composition_details,87433,82.23,High,Retain available descriptions; preserve missin...
2,Samples,JSON,Empty sample_info JSON objects,31561,29.68,Medium,Keep sample records and represent unavailable ...
3,papers,Missing Values,Missing values in issue,3873,6.86,Low,Preserve missing values as null. Evaluate whet...
4,Samples,JSON,Missing sample_info JSON,2314,2.18,Low,Handle missing JSON values safely using json.l...
5,samples,Missing Values,Missing values in sample_info,2314,2.18,Low,Handle missing JSON safely in Python and retai...
6,papers,Missing Values,Missing values in page,2266,4.01,Low,Preserve missing values as null. Evaluate whet...
7,papers,Missing Values,Missing values in ISSN,1901,3.36,Low,Preserve missing values as null. Evaluate whet...
8,papers,Missing Values,Missing values in container_title_short,1575,2.79,Low,Preserve missing values as null. Evaluate whet...
9,papers,Missing Values,Missing values in volume,1451,2.57,Low,Preserve missing values as null. Evaluate whet...


Export Profiling Reports


In [10]:
OUTPUT_DIR = Path("..") / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

reports = {
    "dataset_summary.csv": summary_df,
    "missing_values_report.csv": missing_df,
    "primary_key_report.csv": key_results_df,
    "referential_integrity_report.csv": referential_df,
    "json_validation_report.csv": json_summary,
    "data_quality_issue_register.csv": issue_register
}

for filename, report in reports.items():
    report.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"Exported: {filename}")

print("\nAll profiling reports exported successfully!")

Exported: dataset_summary.csv
Exported: missing_values_report.csv
Exported: primary_key_report.csv
Exported: referential_integrity_report.csv
Exported: json_validation_report.csv
Exported: data_quality_issue_register.csv

All profiling reports exported successfully!
